In [1]:
import os
import glob
import pandas as pd

BASE = r"C:\Users\ariel\Desktop\Thesis\F-TM-CR"
OUT_DIR = os.path.join(BASE, "data", "census_final_data")
os.makedirs(OUT_DIR, exist_ok=True)

# ה-ZIP3 נלקחים מהדאטה המסונן (אחרי 02+03), לא מהקובץ הגולמי
candidates = glob.glob(os.path.join(BASE, "data", "03_advanced_prep", "lc_after_03_advanced_prep_*.csv"))
if not candidates:
    raise FileNotFoundError("No lc_after_03_advanced_prep_*.csv found")
input_path = max(candidates, key=os.path.getmtime)
print(f"Using filtered LC data: {os.path.basename(input_path)}")

df_zip = pd.read_csv(
    input_path,
    usecols=["zip3"],
    dtype={"zip3": "string"},
    low_memory=False
)

df_zip = df_zip.dropna(subset=["zip3"])
df_zip["zip3"] = df_zip["zip3"].str.extract(r"(\d{1,3})", expand=False).str.zfill(3)

zip3_counts = df_zip["zip3"].value_counts().reset_index()
zip3_counts.columns = ["zip3", "count"]

zip3_counts.to_csv(os.path.join(OUT_DIR, "zip3_counts_from_lc.csv"), index=False)
print(f"Loans: {len(df_zip)} | unique ZIP3: {len(zip3_counts)}")

Using filtered LC data: lc_after_03_advanced_prep_basic+test_20260715_2119.csv


Loans: 217287 | unique ZIP3: 843


In [2]:
import os
import requests
import numpy as np
import pandas as pd

API_KEY = os.getenv("CENSUS_API_KEY")
if not API_KEY:
    raise RuntimeError("CENSUS_API_KEY not found in environment variables.")

BASE = r"C:\Users\ariel\Desktop\Thesis\F-TM-CR"
OUT_DIR = os.path.join(BASE, "data", "census_final_data")
os.makedirs(OUT_DIR, exist_ok=True)

# שנתון קבוע: ACS 5-year 2012 = ממוצע 2008-2012, תואם את תקופת ההלוואות (2011-2012).
# הערות זמינות לשנה זו:
#  - השכלה: B15003 (קיימת מ-2012)
#  - שפה: B16002 (המקבילה הישנה של C16002, אותם קודים)
#  - ביטוח בריאות: B27001 (B27010 לא קיימת ב-2012; סוכמים 18 תאי "ללא ביטוח" לפי מין וגיל)
#  - פס רחב (B28002): לא קיים לפני 2017 - השאלה נוספה לסקר רק ב-2013, לכן הושמט
ACS_YEAR = 2012

# 18 תאי "No health insurance coverage" בטבלת B27001 (גברים 005..029, נשים 033..057)
UNINSURED_CODES = {
    f"B27001_{n:03d}E": f"unins_{i:02d}"
    for i, n in enumerate(
        [5, 8, 11, 14, 17, 20, 23, 26, 29, 33, 36, 39, 42, 45, 48, 51, 54, 57], 1
    )
}

VAR_CODES = {
    # --- אוכלוסייה ---
    "B01003_001E": "total_pop",
    "B11001_001E": "total_households",
    # --- גזע (B02001, race alone) ---
    "B02001_001E": "race_total",
    "B02001_002E": "white",
    "B02001_003E": "black",
    "B02001_005E": "asian",
    # --- מוצא היספני + Not-Hispanic לפי גזע (B03002) ---
    "B03002_003E": "white_nh",
    "B03002_004E": "black_nh",
    "B03002_006E": "asian_nh",
    "B03002_012E": "hispanic",
    # --- הגירה (B05002) ---
    "B05002_001E": "nativity_total",
    "B05002_013E": "foreign_born",
    # --- שפת משק בית (B16002; זהה במבנה ל-C16002 המאוחרת) ---
    "B16002_001E": "hh_lang_total",
    "B16002_002E": "hh_english_only",
    "B16002_004E": "hh_ltd_eng_spanish",
    "B16002_007E": "hh_ltd_eng_indoeuro",
    "B16002_010E": "hh_ltd_eng_asian",
    "B16002_013E": "hh_ltd_eng_other",
    # --- הכנסה ועוני ---
    "B19013_001E": "median_income",
    "B17001_001E": "poverty_universe",
    "B17001_002E": "poverty_count",
    # --- תעסוקה (גיל +16, B23025) ---
    "B23025_003E": "civ_labor_force",
    "B23025_005E": "unemployed",
    # --- השכלה (גיל +25, B15003) ---
    "B15003_001E": "edu_total_25plus",
    "B15003_022E": "edu_bachelor",
    "B15003_023E": "edu_master",
    "B15003_024E": "edu_professional",
    "B15003_025E": "edu_doctorate",
    # --- דיור (B25003, B25077) ---
    "B25003_001E": "tenure_total",
    "B25003_002E": "owner_occupied",
    "B25003_003E": "renter_occupied",
    "B25077_001E": "median_home_value",
    # --- ביטוח בריאות (B27001) ---
    "B27001_001E": "hi_universe",
    **UNINSURED_CODES,
}

LTD_ENG_COLS = ["hh_ltd_eng_spanish", "hh_ltd_eng_indoeuro", "hh_ltd_eng_asian", "hh_ltd_eng_other"]
EDU_BACHELOR_PLUS = ["edu_bachelor", "edu_master", "edu_professional", "edu_doctorate"]
UNINSURED_COLS = list(UNINSURED_CODES.values())

# ---------------------------------------------------------------
# משיכה ארצית - כל ה-ZCTA בארה"ב.
# ה-API מוגבל ל-50 משתנים לבקשה, לכן מפצלים לצ'אנקים וממזגים לפי ZCTA.
# ---------------------------------------------------------------
def fetch_chunk(year: int, codes: list) -> pd.DataFrame:
    base_url = f"https://api.census.gov/data/{year}/acs/acs5"
    params = {
        "get": ",".join(codes),
        "for": "zip code tabulation area:*",
        "key": API_KEY
    }
    r = requests.get(base_url, params=params, timeout=300)
    if r.status_code != 200:
        raise RuntimeError(f"Census API error {r.status_code}: {r.text[:300]}")
    data = r.json()
    df = pd.DataFrame(data[1:], columns=data[0])
    df = df.rename(columns={"zip code tabulation area": "zcta"})
    df["zcta"] = df["zcta"].astype(str).str.zfill(5)
    return df.drop(columns=["state"], errors="ignore")

def fetch_all_zctas(year: int, chunk_size: int = 40) -> pd.DataFrame:
    all_codes = list(VAR_CODES.keys())
    chunks = [all_codes[i:i + chunk_size] for i in range(0, len(all_codes), chunk_size)]

    df = None
    for i, chunk in enumerate(chunks, 1):
        print(f"  chunk {i}/{len(chunks)} ({len(chunk)} vars) ...")
        part = fetch_chunk(year, chunk)
        df = part if df is None else df.merge(part, on="zcta", how="outer")

    df = df.rename(columns=VAR_CODES)

    # numeric clean: קודים שליליים של ה-API (annotation values) = NA
    for c in VAR_CODES.values():
        df[c] = pd.to_numeric(df[c], errors="coerce")
        df.loc[df[c] < 0, c] = pd.NA

    # ZCTA -> ZIP3
    df["zip3"] = df["zcta"].str[:3]
    return df

print(f"Fetching all US ZCTAs (ACS5 {ACS_YEAR}) ...")
raw = fetch_all_zctas(ACS_YEAR)
print(f"ZCTAs fetched: {len(raw)}")

# ---------------------------------------------------------------
# אגרגציה ל-ZIP3
# ספירות: סכום. מדיאנים לא מתחברים בסכימה, לכן קירוב בממוצע משוקלל:
# הכנסה חציונית משוקללת במספר משקי הבית של כל ZCTA,
# שווי דירה חציוני משוקלל במספר היחידות בבעלות.
# ---------------------------------------------------------------
COUNT_VARS = [c for c in VAR_CODES.values() if c not in ("median_income", "median_home_value")]

WEIGHTED_MEDIANS = [
    ("median_income",     "total_households", "median_income_approx"),
    ("median_home_value", "owner_occupied",   "median_home_value_approx"),
]

def weighted_mean_by_zip3(df: pd.DataFrame, val: str, w: str, out_name: str) -> pd.DataFrame:
    d = df.dropna(subset=[val, w])
    d = d[(d[w] > 0) & (d[val] > 0)].copy()
    d["_wx"] = d[val] * d[w]
    g = d.groupby("zip3", as_index=False)[["_wx", w]].sum()
    g[out_name] = g["_wx"] / g[w]
    return g[["zip3", out_name]]

zip3_agg = raw.groupby("zip3", as_index=False)[COUNT_VARS].sum(min_count=1)
for val, w, out_name in WEIGHTED_MEDIANS:
    zip3_agg = zip3_agg.merge(weighted_mean_by_zip3(raw, val, w, out_name), on="zip3", how="left")

# ---------------------------------------------------------------
# Feature engineering - שיעורים מתוקננים
# ---------------------------------------------------------------
def safe_ratio(df, num, den):
    r = df[num] / df[den]
    r[df[den].isna() | (df[den] <= 0)] = pd.NA
    return r

zip3_agg["ltd_eng_hh"] = zip3_agg[LTD_ENG_COLS].sum(axis=1, min_count=1)
zip3_agg["bachelor_plus"] = zip3_agg[EDU_BACHELOR_PLUS].sum(axis=1, min_count=1)
zip3_agg["uninsured"] = zip3_agg[UNINSURED_COLS].sum(axis=1, min_count=1)

zip3_agg["share_white"]          = safe_ratio(zip3_agg, "white", "race_total")
zip3_agg["share_black"]          = safe_ratio(zip3_agg, "black", "race_total")
zip3_agg["share_asian"]          = safe_ratio(zip3_agg, "asian", "race_total")
zip3_agg["share_white_nh"]       = safe_ratio(zip3_agg, "white_nh", "race_total")
zip3_agg["share_black_nh"]       = safe_ratio(zip3_agg, "black_nh", "race_total")
zip3_agg["share_asian_nh"]       = safe_ratio(zip3_agg, "asian_nh", "race_total")
zip3_agg["share_hispanic"]       = safe_ratio(zip3_agg, "hispanic", "race_total")
zip3_agg["share_foreign_born"]   = safe_ratio(zip3_agg, "foreign_born", "nativity_total")
zip3_agg["share_non_english_hh"] = 1 - safe_ratio(zip3_agg, "hh_english_only", "hh_lang_total")
zip3_agg["share_ltd_english_hh"] = safe_ratio(zip3_agg, "ltd_eng_hh", "hh_lang_total")
zip3_agg["poverty_rate"]         = safe_ratio(zip3_agg, "poverty_count", "poverty_universe")
zip3_agg["unemployment_rate"]    = safe_ratio(zip3_agg, "unemployed", "civ_labor_force")
zip3_agg["share_bachelor_plus"]  = safe_ratio(zip3_agg, "bachelor_plus", "edu_total_25plus")
zip3_agg["homeownership_rate"]   = safe_ratio(zip3_agg, "owner_occupied", "tenure_total")
zip3_agg["uninsured_rate"]       = safe_ratio(zip3_agg, "uninsured", "hi_universe")

RATE_COLS = [
    "share_white", "share_black", "share_asian",
    "share_white_nh", "share_black_nh", "share_asian_nh", "share_hispanic",
    "share_foreign_born", "share_non_english_hh", "share_ltd_english_hh",
    "poverty_rate", "unemployment_rate", "share_bachelor_plus",
    "homeownership_rate", "uninsured_rate",
]
FEATURE_COLS = RATE_COLS + [
    "median_income_approx", "median_home_value_approx",
    "total_pop", "total_households",
]

# ---------------------------------------------------------------
# סינון לרשימת ה-ZIP3 של LendingClub המסונן + טיפול בחוסרים
# ---------------------------------------------------------------
lc_zip3 = pd.read_csv(os.path.join(OUT_DIR, "zip3_counts_from_lc.csv"), dtype={"zip3": "string"})
lc_zip3["zip3"] = lc_zip3["zip3"].str.zfill(3)
lc_set = set(lc_zip3["zip3"].dropna().unique())

features = zip3_agg[zip3_agg["zip3"].isin(lc_set)][["zip3"] + FEATURE_COLS].copy()
features = features.sort_values("zip3").reset_index(drop=True)

# ZIP3 של LC שאין להם אף ZCTA (בסיסים צבאיים, תיבות דואר וכו') - נשמרים בצד
missing = sorted(lc_set - set(features["zip3"]))
pd.DataFrame({"zip3": missing}).to_csv(os.path.join(OUT_DIR, "missing_zip3_in_panel.csv"), index=False)

# אימפוטציה: NA -> חציון על פני ה-ZIP3 הנכללים
na_counts = features[FEATURE_COLS].isna().sum()
imputed_cols = na_counts[na_counts > 0]
if len(imputed_cols):
    print("Imputed (median) NA counts per column:")
    print(imputed_cols.to_string())
    for c in imputed_cols.index:
        features[c] = features[c].fillna(features[c].median())
else:
    print("No missing values - no imputation needed.")

# עיגול לקריאות: שיעורים ל-4 ספרות, סכומים/מדיאנים למספרים שלמים
features[RATE_COLS] = features[RATE_COLS].round(4)
for c in ["median_income_approx", "median_home_value_approx", "total_pop", "total_households"]:
    features[c] = features[c].round(0).astype("Int64")

assert features[FEATURE_COLS].isna().sum().sum() == 0, "NAs remain after imputation"
assert features["zip3"].is_unique, "Duplicate ZIP3 rows"

features_path = os.path.join(OUT_DIR, f"zip3_census_features_acs5_{ACS_YEAR}.csv")
features.to_csv(features_path, index=False)

coverage = len(features) / len(lc_set) if lc_set else 0
print(f"\nSaved: {features_path}")
print(f"LC unique ZIP3: {len(lc_set)} | with census data: {len(features)} ({coverage:.1%}) | missing: {len(missing)}")
print(features.head())

Fetching all US ZCTAs (ACS5 2012) ...
  chunk 1/2 (40 vars) ...


  chunk 2/2 (11 vars) ...


ZCTAs fetched: 33120
Imputed (median) NA counts per column:
share_white                 2
share_black                 2
share_asian                 2
share_white_nh              2
share_black_nh              2
share_asian_nh              2
share_hispanic              2
share_foreign_born          2
share_non_english_hh        2
share_ltd_english_hh        2
poverty_rate                2
unemployment_rate           2
share_bachelor_plus         2
homeownership_rate          2
uninsured_rate              2
median_income_approx        2
median_home_value_approx    4

Saved: C:\Users\ariel\Desktop\Thesis\F-TM-CR\data\census_final_data\zip3_census_features_acs5_2012.csv
LC unique ZIP3: 843 | with census data: 829 (98.3%) | missing: 14
  zip3  share_white  share_black  share_asian  share_white_nh  share_black_nh  \
0  007       0.6433       0.0727       0.0020          0.0043          0.0007   
1  010       0.9060       0.0242       0.0265          0.8427          0.0215   
2  011       0.55

In [3]:
import os
import pandas as pd

BASE = r"C:\Users\ariel\Desktop\Thesis\F-TM-CR"
OUT_DIR = os.path.join(BASE, "data", "census_final_data")

features = pd.read_csv(os.path.join(OUT_DIR, "zip3_census_features_acs5_2012.csv"), dtype={"zip3": "string"})
features["zip3"] = features["zip3"].str.zfill(3)

# sanity checks
print("Rows (unique ZIP3):", len(features))
print("Duplicated ZIP3:", features["zip3"].duplicated().sum())
print("Total NA:", features.isna().sum().sum())

RATE_COLS = [
    "share_white", "share_black", "share_asian",
    "share_white_nh", "share_black_nh", "share_asian_nh", "share_hispanic",
    "share_foreign_born", "share_non_english_hh", "share_ltd_english_hh",
    "poverty_rate", "unemployment_rate", "share_bachelor_plus",
    "homeownership_rate", "uninsured_rate",
]
for c in RATE_COLS:
    bad = ((features[c] < 0) | (features[c] > 1)).sum()
    if bad:
        print(f"WARNING: {c} out of [0,1] in {bad} rows")

print(features[RATE_COLS + ["median_income_approx", "median_home_value_approx"]].describe().T)

Rows (unique ZIP3): 829
Duplicated ZIP3: 0
Total NA: 0
                          count           mean            std         min  \
share_white               829.0       0.791858       0.159002      0.0337   
share_black               829.0       0.101771       0.124618      0.0006   
share_asian               829.0       0.031476       0.051804      0.0000   
share_white_nh            829.0       0.715675       0.206776      0.0043   
share_black_nh            829.0       0.099214       0.123275      0.0006   
share_asian_nh            829.0       0.031103       0.051326      0.0000   
share_hispanic            829.0       0.118602       0.145223      0.0014   
share_foreign_born        829.0       0.086508       0.092788      0.0010   
share_non_english_hh      829.0       0.157903       0.149150      0.0000   
share_ltd_english_hh      829.0       0.033812       0.047919      0.0000   
poverty_rate              829.0       0.155422       0.055603      0.0371   
unemployment_rate    